# Extraction de features de surface depuis les GPX
Récupère depuis OpenStreetMap : pavés, routes étroites, surface, type de route, etc.
pour chaque fichier GPX de course cycliste.

In [1]:
import os
import time
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
import requests
from shapely.geometry import LineString, Point
from tqdm import tqdm

GPX_DIR = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/gpx_files_2'
OUTPUT_BASE = '/Users/arthurdeletang/Desktop/Stage M1/Code/data/matching'

# Charger les GPX World Tour 2025
df_matching = pd.read_csv(os.path.join(OUTPUT_BASE, 'wt', 'matching_wt_2025.csv'))
df_matching = df_matching[df_matching['statut'] == 'match']
gpx_wt_2025 = df_matching['fichier_gpx'].dropna().unique()
print(f"GPX World Tour 2025 : {len(gpx_wt_2025)}")

/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


GPX World Tour 2025 : 164


In [2]:
# ── 1. Chargement des points GPX ─────────────────────────────────────────────

def charger_points_gpx(filepath):
    """Retourne une liste de (lat, lon) depuis un fichier GPX."""
    tree = ET.parse(filepath)
    root = tree.getroot()
    ns = {'gpx': 'http://www.topografix.com/GPX/1/1'}
    points = root.findall('.//gpx:trkpt', ns)
    return [(float(p.get('lat')), float(p.get('lon'))) for p in points]

In [3]:
# ── 2. Requête Overpass correcte ──────────────────────────────────────────────
#
# Tags OSM pertinents :
#   surface=cobblestone / sett / paving_stones  → pavés
#   surface=unpaved / gravel / dirt / grass     → non-bitumé
#   width=* / lanes=1                           → route étroite
#   highway=track / path                        → chemin/piste
#
# On découpe la trace en segments de ~5 km pour éviter les bbox trop grandes.

OVERPASS_INSTANCES = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://overpass.openstreetmap.ru/api/interpreter',
]

def overpass_request(query, timeout=60):
    """Tente chaque instance Overpass jusqu'à succès."""
    for url in OVERPASS_INSTANCES:
        try:
            r = requests.post(url, data={'data': query}, timeout=timeout)
            if r.status_code == 200:
                return r.json()
        except Exception as e:
            print(f"  {url} → {e}")
    return None


def construire_query_surface(bbox):
    """
    Requête Overpass pour récupérer :
    - tous les ways avec un tag surface (pavés, gravier, asphalte…)
    - les ways avec width ou lanes (pour les routes étroites)
    - les chemins/pistes (highway=track|path)
    """
    s, w, n, e = bbox
    return f"""
[out:json][timeout:60];
(
  way["surface"]({s},{w},{n},{e});
  way["width"]({s},{w},{n},{e});
  way["lanes"]({s},{w},{n},{e});
  way["highway"~"^(track|path|cycleway)$"]({s},{w},{n},{e});
);
out body geom;
"""

In [4]:
# ── 3. Matching géographique OSM ↔ trace GPX ──────────────────────────────────
#
# Pour chaque way OSM, on vérifie si au moins un de ses nœuds est à moins de
# SEUIL_METRES de la trace GPX. Si oui, le way est considéré "sur la route".

SEUIL_METRES = 25  # distance max way OSM ↔ trace (en mètres)
DEG_PAR_METRE = 1 / 111_000  # approximation grossière

def way_sur_trace(way_nodes, trace_line, seuil_m=SEUIL_METRES):
    """
    Retourne True si au moins un nœud du way OSM est à moins de seuil_m
    de la trace GPX (représentée comme un LineString Shapely).
    """
    seuil_deg = seuil_m * DEG_PAR_METRE
    for node in way_nodes:
        pt = Point(node['lon'], node['lat'])
        if trace_line.distance(pt) < seuil_deg:
            return True
    return False


def longueur_way_km(way_nodes):
    """Calcule la longueur approximative d'un way OSM en km (Haversine)."""
    if len(way_nodes) < 2:
        return 0.0
    R = 6371
    total = 0.0
    for i in range(len(way_nodes) - 1):
        lat1, lon1 = np.radians(way_nodes[i]['lat']),   np.radians(way_nodes[i]['lon'])
        lat2, lon2 = np.radians(way_nodes[i+1]['lat']), np.radians(way_nodes[i+1]['lon'])
        dlat, dlon = lat2 - lat1, lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        total += R * 2 * np.arcsin(np.sqrt(a))
    return total

In [5]:
# ── 4. Classification des surfaces ────────────────────────────────────────────

# Tags OSM → catégorie
PAVES = {'cobblestone', 'sett', 'paving_stones', 'cobblestone:flattened'}
NON_BITUME = {'unpaved', 'gravel', 'dirt', 'grass', 'ground', 'sand',
              'compacted', 'fine_gravel', 'woodchips', 'mud'}
BITUME = {'asphalt', 'paved', 'concrete', 'concrete:plates',
          'concrete:lanes', 'tar', 'chipseal'}

def classifier_surface(tags):
    s = tags.get('surface', '').lower()
    if s in PAVES:     return 'pave'
    if s in NON_BITUME: return 'non_bitume'
    if s in BITUME:    return 'bitume'
    if s:              return 'autre_surface'
    hw = tags.get('highway', '')
    if hw in ('track', 'path'): return 'non_bitume'
    return 'inconnu'

def est_route_etroite(tags):
    """
    Heuristique : route étroite si
    - width < 4 m, ou
    - lanes = 1, ou
    - highway = track / path (pas de lignes marquées)
    """
    try:
        w = float(str(tags.get('width', '')).replace('m', '').strip())
        if w < 4:
            return True
    except (ValueError, TypeError):
        pass
    try:
        if int(tags.get('lanes', 99)) == 1:
            return True
    except (ValueError, TypeError):
        pass
    if tags.get('highway') in ('track', 'path'):
        return True
    return False

In [6]:
# ── 5. Extraction complète pour un GPX ────────────────────────────────────────

def extraire_features_surface(filepath, n_segments=8):
    """
    Pour un fichier GPX, interroge OSM par segments et calcule :
      - km_pave           : km de pavés sur la trace
      - km_non_bitume     : km de non-bitumé
      - km_etroit         : km de routes étroites (width<4m ou lanes=1)
      - pct_pave          : % de la distance totale en pavés
      - pct_non_bitume    : idem non-bitumé
      - pct_etroit        : idem routes étroites
      - n_secteurs_pave   : nombre de secteurs pavés distincts
    """
    coords = charger_points_gpx(filepath)
    if len(coords) < 2:
        return None

    # LineString Shapely (lon, lat) pour calcul de distances
    trace_line = LineString([(lon, lat) for lat, lon in coords])

    # Distance totale GPX (haversine)
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371
        dlat, dlon = np.radians(lat2-lat1), np.radians(lon2-lon1)
        a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
        return R * 2 * np.arcsin(np.sqrt(a))

    dist_totale = sum(haversine(coords[i][0], coords[i][1],
                                coords[i+1][0], coords[i+1][1])
                      for i in range(len(coords)-1))

    # Découper la trace en n_segments pour les requêtes OSM
    idx_cuts = np.linspace(0, len(coords)-1, n_segments+1, dtype=int)
    segments = [coords[idx_cuts[i]:idx_cuts[i+1]+1]
                for i in range(n_segments)]

    km_pave = 0.0
    km_non_bitume = 0.0
    km_etroit = 0.0
    secteurs_pave = set()

    for seg in segments:
        if len(seg) < 2:
            continue
        lats = [p[0] for p in seg]
        lons = [p[1] for p in seg]
        bbox = (min(lats)-0.001, min(lons)-0.001,
                max(lats)+0.001, max(lons)+0.001)

        query = construire_query_surface(bbox)
        data = overpass_request(query)
        if data is None:
            continue

        for element in data.get('elements', []):
            if element.get('type') != 'way':
                continue
            nodes = element.get('geometry', [])
            if not nodes:
                continue

            if not way_sur_trace(nodes, trace_line):
                continue  # way hors de la trace → ignoré

            tags = element.get('tags', {})
            longueur = longueur_way_km(nodes)
            surface_cat = classifier_surface(tags)
            etroit = est_route_etroite(tags)
            way_id = element['id']

            if surface_cat == 'pave':
                km_pave += longueur
                secteurs_pave.add(way_id)
            elif surface_cat == 'non_bitume':
                km_non_bitume += longueur

            if etroit:
                km_etroit += longueur

        time.sleep(1)  # politesse envers l'API Overpass

    if dist_totale == 0:
        return None

    return {
        'distance_km':    round(dist_totale, 2),
        'km_pave':        round(km_pave, 2),
        'km_non_bitume':  round(km_non_bitume, 2),
        'km_etroit':      round(km_etroit, 2),
        'pct_pave':       round(100 * km_pave / dist_totale, 2),
        'pct_non_bitume': round(100 * km_non_bitume / dist_totale, 2),
        'pct_etroit':     round(100 * km_etroit / dist_totale, 2),
        'n_secteurs_pave': len(secteurs_pave),
    }

In [7]:
# ── 6. Test sur un seul fichier ───────────────────────────────────────────────

fichier_test = '2025 Santos Tour Down Under Stage 1.gpx'
res = extraire_features_surface(os.path.join(GPX_DIR, fichier_test))
print(res)

  https://overpass.openstreetmap.ru/api/interpreter → HTTPSConnectionPool(host='overpass.openstreetmap.ru', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass.openstreetmap.ru', port=443) at 0x13ef68790>, 'Connection to overpass.openstreetmap.ru timed out. (connect timeout=60)'))


KeyboardInterrupt: 

In [ ]:
# ── 7. Extraction sur tous les GPX WT 2025 ───────────────────────────────────
#
# ATTENTION : chaque course = 8 requêtes Overpass + 1s sleep → ~8s/course
# Pour 30 courses : ~4 min

features_surface = {}

for fname in tqdm(gpx_wt_2025):
    path = os.path.join(GPX_DIR, fname)
    if not os.path.exists(path):
        continue
    try:
        feats = extraire_features_surface(path)
        if feats:
            features_surface[fname] = feats
    except Exception as e:
        print(f"Erreur sur {fname} : {e}")

df_surface = pd.DataFrame.from_dict(features_surface, orient='index').reset_index()
df_surface.rename(columns={'index': 'fichier_gpx'}, inplace=True)

print(df_surface.shape)
df_surface.head(10)

In [ ]:
# ── 8. Sauvegarde ─────────────────────────────────────────────────────────────

output_path = os.path.join(OUTPUT_BASE, 'wt', 'features_surface_wt_2025.csv')
df_surface.to_csv(output_path, index=False)
print(f"Sauvegardé : {output_path}")

# Aperçu des courses avec le plus de pavés
df_surface.sort_values('km_pave', ascending=False).head(5)[[
    'fichier_gpx', 'distance_km', 'km_pave', 'pct_pave', 'n_secteurs_pave'
]]